In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from astropy.constants import c, h, e
import pickle
from scipy.interpolate import interp1d

%matplotlib widget

In [ ]:
data_dir = '/home/lmousset/Projets_Recherche/LSST/Mesure_transmission_tcbp/'

# Photodiode 8D057 measured by Marc

Three photodiodes were measured by Marc : 8D057, 6A057, 6A058.
We have used the 8D057.

In [ ]:
# Photodiode transmission
photodiode_Qeff = np.load(data_dir + 'Photodiodes/preliminary_quantum_efficiency_curves.npy')

print(photodiode_Qeff.shape)

wls = np.zeros(137)
qe_6A057 = np.zeros(137)
qe_6A057_err = np.zeros(137)
for l in range(137):
    wls[l] = photodiode_Qeff[l][0]
    qe_6A057[l] = photodiode_Qeff[l][1]
    qe_6A057_err[l] = photodiode_Qeff[l][2]/100
print(wls)

qe_8D057 = np.zeros(137)
qe_8D057_err = np.zeros(137)
for l in range(137):
    wls[l] = photodiode_Qeff[l][0]
    qe_8D057[l] = photodiode_Qeff[l][5]
    qe_8D057_err[l] = photodiode_Qeff[l][6]/100
print(wls)

In [ ]:
# Plot the 3 photodiodes measured by Marc
plt.figure(figsize=(10, 6))
for l in range(137):
    plt.errorbar(wls[l], photodiode_Qeff[l][1], yerr=photodiode_Qeff[l][2]/100, fmt='.', color='green', label='6A057')
    plt.errorbar(wls[l], photodiode_Qeff[l][3], yerr=photodiode_Qeff[l][4]/100, fmt='.', color='red', label='6A058')
    plt.errorbar(wls[l], photodiode_Qeff[l][5], yerr=photodiode_Qeff[l][6]/100, fmt='.', color='blue', label='8D057')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Quantum Efficiency e-/photon')
plt.show()

In [ ]:
# Interpolation
interpolator_pd_6A057 = interp1d(wls, qe_6A057, kind='cubic', fill_value='extrapolate')
interpolator_pd_8D057 = interp1d(wls, qe_8D057, kind='cubic', fill_value='extrapolate')

wls_interp = np.linspace(360, 1040, 1000)
pd_6A057_interp = interpolator_pd_6A057(wls_interp)
pd_8D057_interp = interpolator_pd_8D057(wls_interp)

plt.figure(figsize=(10, 6))
plt.errorbar(wls, qe_6A057, yerr=None, fmt='.', color='green', label='Measurement 6A057')
plt.plot(wls_interp, pd_6A057_interp, color='blue', label='Cubic Interpolation 6A057')

plt.errorbar(wls, qe_8D057, yerr=None, fmt='.', color='orange', label='Measurement 8D057')
plt.plot(wls_interp, pd_8D057_interp, color='red', label='Cubic Interpolation 8D057')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Quantum efficiency (e-/photon)')
plt.legend()
plt.show()

# Sauvegarder
with open(data_dir + 'Interp_pd_6A057_e_per_photon.pkl', 'wb') as file:
    pickle.dump(interpolator_pd_6A057, file)
with open(data_dir + 'Interp_pd_8D057_e_per_photon.pkl', 'wb') as file:
    pickle.dump(interpolator_pd_8D057, file)

# NIST photodiode 6K096

In [ ]:
# NIST photodiode data
csv_file_NIST = data_dir + 'NIST_6K096.csv'

df_pd_NIST = pd.read_csv(csv_file_NIST)
print(df_pd_NIST.head())

wls_pd_NIST = df_pd_NIST['Wavelength_nm'].values
pd_NIST_AperW = df_pd_NIST['spectral_power_resp_AperW'].values
pd_NIST_error = df_pd_NIST['Relative_uncert_pourcent'].values

In [ ]:
plt.figure(figsize=(10, 6))
plt.errorbar(wls_pd_NIST, pd_NIST_AperW, yerr=pd_NIST_error/100, fmt='.', color='orange')
plt.xlabel('Wavelength (nm)')
plt.ylabel('NIST Photodiode Responsivity (A/W)')

plt.show()

In [ ]:
# Conversion en e-/photon
pd_NIST_e_per_photon = (pd_NIST_AperW * h.value * c.value ) / (wls_pd_NIST * 1e-9 * e.value)

In [ ]:
# Interpolation
interpolator_pd_NIST = interp1d(wls_pd_NIST, pd_NIST_e_per_photon, kind='cubic', fill_value='extrapolate')

wls_interp = np.linspace(300, 1100, 1000)
pd_NIST_e_per_photon_interp = interpolator_pd_NIST(wls_interp)

plt.figure(figsize=(10, 6))
plt.errorbar(wls_pd_NIST, pd_NIST_e_per_photon, yerr=None, fmt='.', color='orange', label='Measurement')
plt.plot(wls_interp, pd_NIST_e_per_photon_interp, color='blue', label='Cubic Interpolation')
plt.xlabel('Wavelength (nm)')
plt.ylabel('NIST Photodiode Responsivity (e-/photon)')
plt.legend()
plt.show()

# Sauvegarder
with open(data_dir + 'Interp_pd_NIST_e_per_photon.pkl', 'wb') as file:
    pickle.dump(interpolator_pd_NIST, file)


In [ ]:
# Charger plus tard
with open(data_dir + 'Interp_pd_NIST_e_per_photon.pkl', 'rb') as file:
    interpolator_pd_NIST = pickle.load(file)

# Lens transmission

In [ ]:
csv_file = data_dir + 'lens_ratio_calibration_system_2.csv'

df_lens = pd.read_csv(csv_file)
print(df_lens.head())

wls_lens = df_lens['Wavelength'].values
transmission_lens = df_lens['lens_ratio'].values
lens_ratio_error = df_lens['ratio_error'].values

In [ ]:
# Interpolation
interpolator_lens = interp1d(wls_lens, transmission_lens, kind='linear', fill_value='extrapolate')

wls_interp = np.linspace(300, 1100, 1000)
transmission_lens_interp = interpolator_lens(wls_interp)

plt.figure(figsize=(10, 6))
plt.errorbar(wls_lens, transmission_lens, yerr=lens_ratio_error, fmt='o', color='m', label='Measurement')
plt.plot(wls_interp, transmission_lens_interp, color='blue', label='Linear Interpolation')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Lens Transmission')
plt.legend()
plt.show()

# Sauvegarder
with open(data_dir + 'Interp_lens_transmission.pkl', 'wb') as file:
    pickle.dump(interpolator_lens, file)